In [ ]:
# --- Section 1: Header ---
# Package definition + metadata + zero-cost policy statement (M2-P0-13).
import json, os, sys

PACKAGE = json.loads(r'''
__GHARIBO_PACKAGE_JSON__
''')

print('=' * 72)
print('GHARIBO AI LAB - Training Package')
print('=' * 72)
print('experiment_id  :', PACKAGE['experiment_id'])
print('package_id     :', PACKAGE['package_id'])
print('schema_version :', PACKAGE['schema_version'])
print('base_model     :', PACKAGE['base_model'], '@', PACKAGE['base_model_revision'])
print('loader_model   :', PACKAGE['loader_model_id'])
print('dataset_hash   :', PACKAGE['dataset']['dataset_hash'])
print('engine         :', PACKAGE['engine']['engine'], PACKAGE['engine']['engine_version'])
print('dtype/seq_len  :', PACKAGE['dtype'], '/', PACKAGE['sequence_length'])
print('')
print('ZERO-COST POLICY: this run executes on the free Kaggle tier only.')
print('No paid training provider and no paid storage are used anywhere.')

# The package declares evaluation intent only - no metrics are ever claimed here.
assert PACKAGE['evaluation_config']['executed'] is False
assert PACKAGE['evaluation_config']['status'] == 'NOT_RUN'
print('evaluation_config: NOT_RUN (declared intent only)')


In [ ]:
# --- Section 2: Hardware detect ---
# Print GPU name, compute capability, VRAM and selected dtype (M2-P0-05).
import torch

print('torch:', torch.__version__)
assert torch.cuda.is_available(), 'No CUDA GPU detected - enable a Kaggle GPU accelerator.'
gpu_name = torch.cuda.get_device_name(0)
cap_major, cap_minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
compute_capability = 'sm_%d%d' % (cap_major, cap_minor)
# T4 = Turing (sm_75): bf16 is unsupported; fp16 is the only tensor-core dtype.
selected_dtype = 'fp16' if cap_major < 8 else 'bf16'
print('GPU            :', gpu_name)
print('compute cap    :', compute_capability)
print('VRAM (GB)      :', round(vram_gb, 1))
print('selected dtype :', selected_dtype)


In [ ]:
# --- Section 3: Budget gate - fail loudly BEFORE training (M2-P0-05, Q2) ---
MIN_VRAM_GB = 14.0
assert vram_gb >= MIN_VRAM_GB, (
    'Insufficient VRAM: %.1f GB < %.1f GB required for gpt-oss-20b QLoRA. '
    'Aborting before training rather than OOM-ing mid-run.' % (vram_gb, MIN_VRAM_GB)
)
assert PACKAGE['dtype'] == 'fp16', 'Package dtype must be fp16 (T4 is Turing/sm_75; bf16 unsupported).'
assert selected_dtype == 'fp16', 'This GPU is not Turing-class; the pinned recipe expects fp16 on a T4.'

max_seq_length = PACKAGE['sequence_length']
if vram_gb < 15.0 and max_seq_length > 512:
    print('VRAM %.1f GB < 15 GB - downgrading max_seq_length %d -> 512' % (vram_gb, max_seq_length))
    max_seq_length = 512
print('budget gate passed; max_seq_length =', max_seq_length)


In [ ]:
# --- Section 4: Install the pinned engine set via uv (M2-P0-06) ---
import os, subprocess, sys

pip_specs = [d['spec'] for d in PACKAGE['engine']['dependencies'] if d['source'] == 'pip']
git_specs = []
for d in PACKAGE['engine']['dependencies']:
    if d['source'] != 'git':
        continue
    spec = d['spec']
    if spec.startswith('@git+') or spec.startswith('git+'):
        git_specs.append(spec.lstrip('@'))
    elif d.get('url'):
        git_specs.append('git+' + d['url'] + spec)
    else:
        git_specs.append(spec.lstrip('@'))
install_args = pip_specs + git_specs
print('Installing pinned set:')
for a in install_args:
    print('  ', a)
# A Kaggle image has no active virtualenv, so `uv pip install` needs an explicit
# target; without one uv aborts with "no virtual environment found".
uv_target = ['--python', sys.executable]
if not os.environ.get('VIRTUAL_ENV'):
    uv_target = ['--system'] + uv_target
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '-qqq', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', *uv_target, '-qqq', *install_args], check=True)
print('install complete')


In [ ]:
# --- Section 5: Verify pinned versions + import smoke test (M2-P0-06, Q10) ---
import importlib
from importlib.metadata import version as pkg_version, PackageNotFoundError

mismatches = []
for d in PACKAGE['engine']['dependencies']:
    if d['source'] != 'pip':
        continue
    name = d['name']
    try:
        installed = pkg_version(name)
    except PackageNotFoundError:
        mismatches.append('%s: not installed (spec %s)' % (name, d['spec']))
        continue
    resolved = d.get('resolved_version')
    if resolved:
        if installed != resolved:
            mismatches.append('%s: installed %s != pinned %s' % (name, installed, resolved))
    else:
        floor = d['spec'].split('>=')[1] if '>=' in d['spec'] else None
        if floor and installed.split('+')[0] < floor:
            mismatches.append('%s: installed %s < floor %s' % (name, installed, floor))
        print('recorded %s==%s (spec %s)' % (name, installed, d['spec']))
assert not mismatches, 'Pinned dependency mismatch: ' + '; '.join(mismatches)
import torch, triton
print('import smoke test ok:', torch.__version__, triton.__version__)


In [ ]:
# --- Section 6: Dataset + split hash verification - hard-fail on mismatch (M2-P0-07) ---
import hashlib, pathlib

WORKING = pathlib.Path('/kaggle/working')
DATA_DIR = WORKING / 'dataset'
if not DATA_DIR.exists():
    DATA_DIR = pathlib.Path('dataset')

def sha256_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

split_lines = {}
for name in ('train', 'validation', 'test'):
    with open(DATA_DIR / (name + '.jsonl'), 'r', encoding='utf-8') as fh:
        split_lines[name] = [ln for ln in fh.read().split('\n') if ln.strip() != '']

all_hashes = []
for name, lines in split_lines.items():
    line_hashes = sorted(sha256_text(ln) for ln in lines)
    actual = sha256_text('\n'.join(line_hashes))
    expected = PACKAGE['dataset']['split_hashes'][name]
    assert actual == expected, 'Split hash mismatch for %s: %s != %s' % (name, actual, expected)
    all_hashes.extend(line_hashes)

dataset_hash = sha256_text('\n'.join(sorted(all_hashes)))
assert dataset_hash == PACKAGE['dataset']['dataset_hash'], (
    'Dataset hash mismatch: %s != %s' % (dataset_hash, PACKAGE['dataset']['dataset_hash'])
)
print('dataset + split hashes verified')
print('  train=%d validation=%d test=%d' % (
    len(split_lines['train']), len(split_lines['validation']), len(split_lines['test'])))


In [ ]:
# --- Section 7: Record -> Harmony mapping (analysis NEVER printed) (M2-P0-08) ---
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(PACKAGE['loader_model_id'])
reasoning_effort = PACKAGE['harmony']['reasoning_effort']
hidden_channels = set(PACKAGE['harmony']['hidden_channels'])
assert 'analysis' in hidden_channels, 'Package must declare the analysis channel as hidden'

def to_harmony_messages(record):
    msgs = []
    msgs.append({'role': 'developer', 'content': 'Produce a well-structured, accurate answer grounded in the provided input and context.'})
    user = record.get('input') or ''
    if record.get('context'):
        user = user + '\n\nContext:\n' + record['context']
    msgs.append({'role': 'user', 'content': user})
    if record.get('reasoning'):
        msgs.append({'role': 'assistant', 'channel': 'analysis', 'content': record['reasoning']})
    final = record.get('chosen_output') or record.get('expected_output')
    if final is not None:
        msgs.append({'role': 'assistant', 'channel': 'final', 'content': final})
    return msgs

def render_texts(records):
    texts = []
    for i, rec in enumerate(records):
        msgs = to_harmony_messages(rec)
        texts.append(tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False, reasoning_effort=reasoning_effort))
        # The rendered text CONTAINS the analysis channel. It is used for training only
        # and is never printed - only the record index is logged.
        if i % 25 == 0:
            print('rendered record index', i)
    return texts

train_records = [json.loads(ln) for ln in split_lines['train']]
train_texts = render_texts(train_records)
print('rendered', len(train_texts), 'training texts (analysis channel hidden from output)')


In [ ]:
# --- Section 8: Load the gpt-oss 4-bit representation (M2-P0-08) ---
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=PACKAGE['loader_model_id'],
    dtype=None,            # auto-detect -> resolves to fp16 on a T4
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    full_finetuning=False,
)
print('model loaded:', PACKAGE['loader_model_id'])


In [ ]:
# --- Section 9: QLoRA adapters - r/alpha/target_modules FROM the package (M2-P0-09) ---
lora = PACKAGE['lora']
model = FastLanguageModel.get_peft_model(
    model,
    r=lora['r'],
    target_modules=list(lora['target_modules']),
    lora_alpha=lora['alpha'],
    lora_dropout=lora['dropout'],
    bias=lora['bias'],
    use_gradient_checkpointing='unsloth',
    random_state=PACKAGE['seed'],
    use_rslora=False,
    loftq_config=None,
)
print('LoRA r=%d alpha=%d modules=%s' % (lora['r'], lora['alpha'], lora['target_modules']))


In [ ]:
# --- Section 10: SFT config from the package (M2-P0-09, M2-P0-10) ---
from trl import SFTConfig, SFTTrainer
from datasets import Dataset as HFDataset

cp = PACKAGE['checkpoint_policy']
sft_kwargs = dict(
    per_device_train_batch_size=PACKAGE['batch']['per_device_train_batch_size'],
    gradient_accumulation_steps=PACKAGE['batch']['gradient_accumulation_steps'],
    warmup_steps=PACKAGE['warmup_steps'],
    learning_rate=PACKAGE['learning_rate'],
    logging_steps=1,
    optim=PACKAGE['optimizer'],
    weight_decay=PACKAGE['weight_decay'],
    lr_scheduler_type=PACKAGE['lr_scheduler_type'],
    seed=PACKAGE['seed'],
    output_dir=str(WORKING / 'outputs'),
    report_to='none',
    save_strategy=cp['save_strategy'],
    save_steps=cp['save_steps'],
    save_total_limit=cp['save_total_limit'],
    fp16=(PACKAGE['dtype'] == 'fp16'),
    bf16=False,
)
if PACKAGE['epochs'] is not None:
    sft_kwargs['num_train_epochs'] = PACKAGE['epochs']
if PACKAGE['max_steps'] is not None:
    sft_kwargs['max_steps'] = PACKAGE['max_steps']

train_dataset = HFDataset.from_dict({'text': train_texts})
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_dataset, args=SFTConfig(**sft_kwargs))
print('SFT config ready (save_strategy=%s, save_steps=%s, save_total_limit=%s)' % (
    cp['save_strategy'], cp['save_steps'], cp['save_total_limit']))


In [ ]:
# --- Section 11: Resume from checkpoint when supplied by the package (M2-P0-11) ---
resume_from_checkpoint = PACKAGE['checkpoint_policy']['resume_from_checkpoint']
if resume_from_checkpoint:
    print('RESUMING from checkpoint:', resume_from_checkpoint)
else:
    print('fresh run - no resume point supplied')


In [ ]:
# --- Section 12: Train (M2-P0-10) ---
train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)
print('training finished')


In [ ]:
# --- Section 13: Save adapter + trainer state + metrics + manifest (M2-P0-10, M2-P0-20) ---
ADAPTER_DIR = WORKING / 'adapter'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()
trainer.save_model(str(WORKING / 'outputs' / 'final'))

metrics = getattr(trainer.state, 'log_history', [])
with open(WORKING / 'metrics.json', 'w', encoding='utf-8') as fh:
    json.dump(metrics, fh, indent=2, sort_keys=True)

manifest = dict(PACKAGE)
manifest['environment_metadata'] = {
    'os': os.name,
    'python_version': sys.version.split()[0],
    'packages': {'torch': torch.__version__, 'triton': triton.__version__},
    'gpu': gpu_name,
    'cuda': torch.version.cuda,
}
manifest['resume_from_checkpoint'] = resume_from_checkpoint
with open(WORKING / 'manifest.json', 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)
print('artifacts saved under', WORKING)


In [ ]:
# --- Section 14: CHECKSUMS.sha256 - per-file + rollup (Q9) ---
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

entries = []
for path in sorted(WORKING.rglob('*')):
    if path.is_file() and path.name != 'CHECKSUMS.sha256':
        rel = path.relative_to(WORKING).as_posix()
        entries.append((rel, sha256_file(path)))
rollup = sha256_text('\n'.join(sorted('%s\t%s' % (rel, digest) for rel, digest in entries)))
lines_out = ['%s  %s' % (digest, rel) for rel, digest in entries]
lines_out.append('# rollup  ' + rollup)
with open(WORKING / 'CHECKSUMS.sha256', 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(lines_out) + '\n')
print('CHECKSUMS.sha256 written; rollup =', rollup)


In [ ]:
# --- Section 15: Optional PRIVATE HF upload via Kaggle Secrets (M2-P0-12) ---
# The token is read by NAME and never printed, never written to any file.
dest = PACKAGE.get('artifact_destination')
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    secret_name = (dest or {}).get('token_secret_name') or 'HF_TOKEN'
    hf_token = UserSecretsClient().get_secret(secret_name)
except Exception:
    hf_token = None

if dest and dest.get('kind') == 'hf' and dest.get('private') is True and hf_token:
    model.push_to_hub_merged(dest['repo_id'], tokenizer=tokenizer, token=hf_token, save_method='mxfp4')
    print('uploaded adapter to private HF repo:', dest['repo_id'])
else:
    print('no HF destination configured (or secret absent) - local /kaggle/working export is the result')
del hf_token


In [ ]:
# --- Section 16: Finalize - completion marker + logs under /kaggle/working (M2-P0-19, M2-P0-20) ---
env_meta = manifest['environment_metadata']
with open(WORKING / 'training.log', 'a', encoding='utf-8') as fh:
    fh.write('experiment_id=' + PACKAGE['experiment_id'] + '\n')
    fh.write('package_id=' + PACKAGE['package_id'] + '\n')
    fh.write('gpu=' + str(env_meta['gpu']) + '\n')
    fh.write('cuda=' + str(env_meta['cuda']) + '\n')
    fh.write('status=COMPLETED\n')
with open(WORKING / 'COMPLETED', 'w', encoding='utf-8') as fh:
    fh.write(PACKAGE['package_id'] + '\n')
print('run complete - outputs persisted under /kaggle/working')
print('NOTE: run as a committed / Save-Version notebook so /kaggle/working persists.')
